In [1]:
from pathlib import Path

import pandas as pd
import networkx as nx

import numpy as np
import pandas as pd
import networkx as nx

In [2]:
# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------

# To avoid path explosion, start with a modest cutoff.
# Set to None to allow arbitrary path lengths.
PATH_CUTOFF = 10

In [3]:
# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------

csv_file = Path("breast_cancer/confidence_rank_unique.csv")  # Change if needed

df = pd.read_csv(csv_file)

In [4]:
# Keep only retained edges
df = df[df["is_kept"] == "KEPT"].copy()

print(f"Kept edges: {len(df)}")

Kept edges: 519


In [5]:
# ------------------------------------------------------------------
# Build graph
# ------------------------------------------------------------------

G = nx.DiGraph()

attribute_columns = [
    c for c in df.columns
    if c not in {"Regulator", "Target", "is_kept"}
]

for _, row in df.iterrows():
    attrs = {c: row[c] for c in attribute_columns}
    G.add_edge(row["Regulator"], row["Target"], **attrs)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 200
Edges: 519


In [6]:
# ------------------------------------------------------------------
# Sign conversion helpers
# ------------------------------------------------------------------

SIGN_TO_NUM = {
    "positive": 1,
    "negative": -1,
}

NUM_TO_SIGN = {
    1: "positive",
    -1: "negative",
}

In [7]:
# ------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------

def replacement_paths(G, source, target, cutoff=None):
    """Generate replacement paths for a removed edge."""

    if source != target:
        yield from nx.all_simple_paths(
            G,
            source=source,
            target=target,
            cutoff=cutoff,
        )
    else:
        for succ in G.successors(source):
            for path in nx.all_simple_paths(
                G,
                source=succ,
                target=source,
                cutoff=cutoff-1,
            ):
                yield [source] + path

In [8]:
# ------------------------------------------------------------------
# Analyze removed edges
# ------------------------------------------------------------------

removed_df = pd.read_csv(csv_file)
removed_df = removed_df[removed_df["is_kept"] == "TAKEN_OUT"].copy()

results = []
all_path_stats = {}

for idx, row in removed_df.iterrows():

    source = row["Regulator"]
    target = row["Target"]
    original_sign = row["Sign"]
    original_rank = row["Confidence rank"]

    print(f"Analyzing removed edge: {source} -> {target}")

    # --------------------------------------------------------------
    # Enumerate all replacement paths
    # --------------------------------------------------------------

    path_stats = []
    current_cutoff = 1

    while len(path_stats) == 0 and current_cutoff <= PATH_CUTOFF:

        print(
            f"  Searching for alternative paths with cutoff={current_cutoff}..."
        )
        
        for path in replacement_paths(
            G,
            source=source,
            target=target,
            cutoff=current_cutoff,
        ):

            edges = list(zip(path[:-1], path[1:]))
            attrs = [G[u][v] for u, v in edges]

            length = len(edges)
            rank = max(a["Confidence rank"] for a in attrs)

            sign_num = 1
            for a in attrs:
                sign_num *= SIGN_TO_NUM[a["Sign"]]

            if sign_num == SIGN_TO_NUM[original_sign] and rank <= original_rank:
                path_stats.append({
                    "path": path,
                    "length": length,
                    "rank": rank,
                    "sign_num": sign_num,
                    "sign": NUM_TO_SIGN[sign_num],
                })

        current_cutoff += 1

    if len(path_stats) == 0:
        print(
            f"No alternative path with same sign found for removed edge "
            f"{source} -> {target}, even with cutoff={PATH_CUTOFF}."
        )

    all_path_stats[(source, target)] = path_stats

    # --------------------------------------------------------------
    # Select best-rank paths
    # --------------------------------------------------------------

    if len(path_stats) == 0:
        results.append({
            "Regulator": source,
            "Target": target,
            "Original Sign": original_sign,
            "Original Rank": original_rank,
            "No Matching Paths": True,
            "Best Rank": None,
            "Number Best Paths": 0,
            "Min Length": None,
            "Max Length": None,
            "Avg Length": None,
        })
        continue

    best_rank = min(p["rank"] for p in path_stats)
    best_paths = [p for p in path_stats if p["rank"] == best_rank]
    lengths = [p["length"] for p in best_paths]

    results.append({
        "Regulator": source,
        "Target": target,
        "Original Sign": original_sign,
        "Original Rank": original_rank,
        "No Matching Paths": False,
        "Best Rank": best_rank,
        "Number Best Paths": len(best_paths),
        "Min Length": int(np.min(lengths)),
        "Max Length": int(np.max(lengths)),
        "Avg Length": float(np.mean(lengths)),
    })

Analyzing removed edge: ABL1 -> MYOD1
  Searching for alternative paths with cutoff=1...
  Searching for alternative paths with cutoff=2...
  Searching for alternative paths with cutoff=3...
  Searching for alternative paths with cutoff=4...
  Searching for alternative paths with cutoff=5...
  Searching for alternative paths with cutoff=6...
Analyzing removed edge: AKT -> AKT
  Searching for alternative paths with cutoff=1...
  Searching for alternative paths with cutoff=2...
  Searching for alternative paths with cutoff=3...
  Searching for alternative paths with cutoff=4...
Analyzing removed edge: AKT -> CyclinD
  Searching for alternative paths with cutoff=1...
  Searching for alternative paths with cutoff=2...
Analyzing removed edge: AKT -> CyclinD_2
  Searching for alternative paths with cutoff=1...
  Searching for alternative paths with cutoff=2...
Analyzing removed edge: AKT -> FAS
  Searching for alternative paths with cutoff=1...
  Searching for alternative paths with cutoff=2

In [9]:
# ------------------------------------------------------------------
# Result table
# ------------------------------------------------------------------

analysis_df = pd.DataFrame(results)

print()
print(f"Analyzed {len(analysis_df)} removed edges.")
print(f"Flagged edges: {analysis_df['No Matching Paths'].sum()}")

analysis_df


Analyzed 189 removed edges.
Flagged edges: 0


,Regulator,Target,Original Sign,Original Rank,No Matching Paths,Best Rank,Number Best Paths,Min Length,Max Length,Avg Length
0,ABL1,MYOD1,negative,4,False,4,1,6,6,6.0
1,AKT,AKT,positive,2,False,1,8,4,4,4.0
2,AKT,CyclinD,positive,2,False,1,1,2,2,2.0
3,AKT,CyclinD_2,positive,2,False,1,1,2,2,2.0
4,AKT,FAS,negative,4,False,1,2,3,3,3.0
...,...,...,...,...,...,...,...,...,...,...
184,pRB,E2F,positive,2,False,1,1,2,2,2.0
185,pertuzumab,EGFR,negative,2,False,1,1,2,2,2.0
186,pertuzumab,ERK,positive,2,False,1,24,9,9,9.0
187,pertuzumab,ERK_2,positive,2,False,1,24,9,9,9.0


In [10]:
missing_match = analysis_df[
    analysis_df["No Matching Paths"]
]

print(missing_match)

Empty DataFrame
Columns: [Regulator, Target, Original Sign, Original Rank, No Matching Paths, Best Rank, Number Best Paths, Min Length, Max Length, Avg Length]
Index: []


In [11]:
pairs = list(
    missing_match.loc[:, ["Regulator", "Target"]
    ].itertuples(index=False, name=None)
)

In [12]:
print(len(pairs))
print(pairs)

0
[]
